In [1]:
import pandas as pd
from math import cos, radians, sqrt
import mpmath as mp
from sklearn.neighbors import BallTree
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from tqdm import tqdm
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import warnings
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 2)
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
tqdm.pandas()

In [3]:
raw = "C:\\Users\\Taavi\\Desktop\\BPhil\\Raw data\\"
clean = "C:\\Users\\Taavi\\Desktop\\BPhil\\Clean data\\"

In [5]:
parcels = pd.read_csv(clean + 'clean_parcels.csv')

In [6]:
geojson = gpd.read_file(raw + 'Pittsburgh_Boundary.geojson')

boundary = geojson.geometry.iloc[2]
# this is the perimeter of the city

polygons = [geom for geom in boundary.geoms]

In [7]:
city = geojson.dissolve()
mpoly = city.geometry.iloc[0]

lngs = []
lats = []

for poly in mpoly.geoms:
    for x, y in poly.exterior.coords:
        lngs.append(x)
        lats.append(y)

bounds = pd.DataFrame({
    'lat': lats,
    'lng': lngs
})

In [13]:
def calc_haversine(lat_row, lng_row, lat_point, lng_point):
    earth_radius = 6_378_137

    lat_row, lng_row, lat_point, lng_point = map(np.radians, [lat_row, lng_row, lat_point, lng_point])

    dist_lat = lat_point - lat_row
    dist_lng = lng_point - lng_row

    a = np.sin(dist_lat / 2) ** 2 + np.cos(lat_row) * np.cos(lat_point) * np.sin(dist_lng / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return earth_radius * c

In [14]:
def return_distances(row, bounds):
    global south_sub, nearest_south_index, nearest_south_point

    north_sub = bounds.loc[bounds['lat'] > row['lat']].reset_index(drop = True)
    east_sub = bounds.loc[bounds['lng'] > row['lng']].reset_index(drop = True)
    south_sub = bounds.loc[bounds['lat'] < row['lat']].reset_index(drop = True)
    west_sub = bounds.loc[bounds['lng'] < row['lng']].reset_index(drop = True)

    if north_sub.empty:
        dist_north = 0
    else:
        nearest_north_index = np.abs(north_sub['lng'] - row['lng']).idxmin()
        nearest_north_point = north_sub.iloc[nearest_north_index]
        dist_north = calc_haversine(row['lat'], row['lng'], nearest_north_point['lat'], nearest_north_point['lng'])

    if east_sub.empty:
        dist_east = 0
    else:
        nearest_east_index = np.abs(east_sub['lat'] - row['lat']).idxmin()
        nearest_east_point = east_sub.iloc[nearest_east_index]
        dist_east = calc_haversine(row['lat'], row['lng'], nearest_east_point['lat'], nearest_east_point['lng'])
    
    if south_sub.empty:
        dist_south = 0
    else:
        nearest_south_index = np.abs(south_sub['lng'] - row['lng']).idxmin()
        nearest_south_point = south_sub.iloc[nearest_south_index]
        dist_south = calc_haversine(row['lat'], row['lng'], nearest_south_point['lat'], nearest_south_point['lng'])
    
    if west_sub.empty:
        dist_west = 0
    else:
        nearest_west_index = np.abs(west_sub['lat'] - row['lat']).idxmin()
        nearest_west_point = west_sub.iloc[nearest_west_index]
        dist_west = calc_haversine(row['lat'], row['lng'], nearest_west_point['lat'], nearest_west_point['lng'])

    return dist_north, dist_east, dist_south, dist_west

In [15]:
parcels[['dist_north', 'dist_east', 'dist_south', 'dist_west']] = parcels.progress_apply(lambda x: pd.Series(return_distances(x, bounds)), axis = 1)

100%|██████████| 144031/144031 [06:32<00:00, 366.73it/s]


In [16]:
parcels.to_csv(clean + 'parcels_distances_to_boundary.csv', index = False)

In [4]:
parcels = pd.read_csv(clean + 'parcels_distances_to_boundary.csv')

### Move forward only with the parcels that are at most 750m from at least one of the four boundary points

In [10]:
def generate_points(row, radius):

    lat = row['lat']
    lng = row['lng']

    conversion_lat = 111111.111
    conversion_lng = 111111.111 * np.cos(np.radians(lat))

    lat_adjustment = radius / conversion_lat
    lng_adjustment = radius / conversion_lng

    north_east = np.array((lng + lng_adjustment, lat + lat_adjustment))
    south_east = np.array((lng + lng_adjustment, lat - lat_adjustment))
    south_west = np.array((lng - lng_adjustment, lat - lat_adjustment))
    north_west = np.array((lng - lng_adjustment, lat + lat_adjustment))

    edge_ne_se = np.linspace(north_east, south_east, int(radius / 10))
    edge_nw_sw = np.linspace(north_west, south_west, int(radius / 10))

    lats = []
    lngs = []

    for i in range(len(edge_ne_se)):
        points = np.linspace(edge_ne_se[i], edge_nw_sw[i], int(radius / 10))

        lats.extend([point[1] for point in points])
        lngs.extend([point[0] for point in points])

    coords = pd.DataFrame({'lat': lats, 'lng': lngs})
    coords_radians = np.radians(coords.values)

    tree = BallTree(coords_radians, metric = 'haversine')

    radius_radians = radius / 6_378_137 # earth's radius

    parcel_coords = np.radians([[row['lat'], row['lng']]])
    idxs = tree.query_radius(parcel_coords, r = radius_radians)[0]
    
    coords = coords.iloc[idxs].reset_index(drop = True)

    return coords

In [11]:
def filterData(data, mpoly):

    global test

    test = data

    filteredData = []

    data.apply(
        lambda x:
        filteredData.append(x) if Point(x['lng'], x['lat']).within(mpoly) else None,
        axis = 1
    )

    return len(filteredData) / len(data)

In [8]:
subset = parcels.copy()
subset = subset.loc[
    (subset['dist_north'] < 750) |
    (subset['dist_east'] < 750) |
    (subset['dist_south'] < 750) |
    (subset['dist_west'] < 750)
]

In [12]:
geojson = gpd.read_file(raw + 'Pittsburgh_Boundary.geojson')

boundary = geojson.geometry.iloc[2]
# this is the perimeter of the city

#polygons = [geom for geom in boundary.geoms]

city = geojson.dissolve()
mpoly = city.geometry.iloc[0]
exterior = mpoly.geoms[0].exterior

results = subset.progress_apply(lambda x: filterData(generate_points(x, 750), mpoly), axis = 1)
#results2 = parcels.iloc[:1000].progress_apply(lambda x: filterData(generate_points(x, 800), mpoly), axis = 1)

100%|██████████| 28776/28776 [2:15:02<00:00,  3.55it/s]  


In [13]:
subset['prop_in_city'] = results

In [24]:
parcels = parcels.merge(subset[['parcelID', 'prop_in_city']], on = 'parcelID', how = 'left').fillna(value = {'prop_in_city': 1.0})

In [44]:
parcels['prop_in_city'].value_counts(normalize = True)

prop_in_city
1.00   0.80
0.89   0.00
0.87   0.00
0.80   0.00
0.84   0.00
       ... 
0.35   0.00
0.30   0.00
0.33   0.00
0.33   0.00
0.26   0.00
Name: proportion, Length: 3077, dtype: float64

In [45]:
parcels.to_csv(clean + 'parcels_prop_in_city.csv', index = False)

### Merging this back onto the original parcels dataframe

In [46]:
parcels = pd.read_csv(clean + 'parcels_prop_in_city.csv')

In [1]:
# toPlot = parcels.query('prop_in_city < 1')

# fig = px.scatter_mapbox(toPlot, lat = 'lat', lon = 'lng', zoom = 10, hover_data = ['nbrhd', 'prop_in_city'])
# fig.update_layout(mapbox_style = 'open-street-map')
# fig.show()

### Continuing... trying to get annuli-specific adjustment values

In [4]:
parcels = pd.read_csv(clean + 'parcels_prop_in_city.csv')

In [5]:
full_area = np.pi * (750**2)

parcels['area_inside'] = full_area * parcels['prop_in_city']
parcels['area_outside'] = full_area - parcels['area_inside']

In [6]:
def calc_implied_distance(row, radius):
    A = row['area_outside']
    R = radius

    f = lambda d: R**2 * mp.acos(d/R) - d * mp.sqrt(R**2 - d**2) - A
    return mp.findroot(f, R/2) # iterate for d such that f(d) == 0, start with guess R/2

In [7]:
parcels['implied_distance_to_flat_boundary'] = parcels.progress_apply(lambda x: calc_implied_distance(x, 750), axis = 1)

100%|██████████| 144031/144031 [02:54<00:00, 825.77it/s] 


In [8]:
parcels['implied_distance_to_flat_boundary'] = parcels['implied_distance_to_flat_boundary'].astype(float)

#### Commentary
This whole process is necessary. We can't just take the distances to the north/east/south/west boundaries and use that as 'd'. In reality very few parcels, if any, are against only a single edge, perpendicular to a cardinal direction. We have to do the area calculation and the implied distance calculation to get an appropriate distance 'd'.

In [9]:
radii = np.arange(50, 800, 50)

In [10]:
def calc_annuli_adjustments(row, radii, threshold_min):
    d = row['implied_distance_to_flat_boundary']

    global last_area_in
    
    adjustments = []

    for R in radii:

        if d > 0:

            if d > R:
                adjustments.append(1)
                continue

            else:
                ratio = d/R

                area_out = R**2 * np.arccos(ratio) - d * np.sqrt(R**2 - d**2)
                full_area = np.pi * R**2
                area_in = full_area - area_out
                area_sub = np.pi * (R-50)**2
                
                if (R - d) < 50:
                    annuli_area_in = area_in - area_sub
                else:
                    annuli_area_in = area_in - last_area_in

                annuli_full_area = full_area - area_sub
                annuli_percent_in = annuli_area_in / annuli_full_area

                adj = max(threshold_min, annuli_percent_in)

                adjustments.append(adj)

                last_area_in = area_in
                continue

        else:

            if np.abs(d) > R:
                adjustments.append(threshold_min) 
                continue

            else:
                ratio = np.abs(d) / R
                #ratio = max(-1, min(1, ratio))

                area_out = R**2 * np.arccos(ratio) - np.abs(d) * np.sqrt(R**2 - np.abs(d)**2)
                full_area = np.pi * R**2
                area_in = full_area - area_out
                area_sub = np.pi * (R-50)**2

                if (R - np.abs(d)) < 50:
                    annuli_area_in = area_in - area_sub
                else:
                    annuli_area_in = area_in - last_area_in
                    
                annuli_full_area = full_area - area_sub
                annuli_percent_in = annuli_area_in / annuli_full_area

                # flip because d is actually negative
                true_annuli_percent_in = 1 - annuli_percent_in

                adj = max(threshold_min, true_annuli_percent_in)

                adjustments.append(adj)

                last_area_in = area_in
                continue

    return adjustments

In [11]:
adj_cols = [f'adjustment_{R}' for R in radii]

parcels[adj_cols] = parcels.progress_apply(lambda row: pd.Series(calc_annuli_adjustments(row, radii, 0.25)), axis = 1)

# threshold_min of 0.25 is arbitrary, not a perfect solution. this probably underadjusts, which I feel is appropriate because incidence of blight
# should arguably be less severe outside the city than in. I would rather underadjust than overadjust
# this is to avoid dividing anything by so extreme a decimal that it blows up
# also this is basically saying the most we can ever adjust is by 4x
# i'm being conservative and forcing an underestimation rather than an overestimation

  0%|          | 0/144031 [00:00<?, ?it/s]

100%|██████████| 144031/144031 [00:28<00:00, 5068.58it/s]


In [12]:
parcels.to_csv(clean + 'parcels_with_adjustments.csv', index = False)